In [ ]:
from snowflake.snowpark import Session
from snowflake.snowpark.functions import col
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import json

In [ ]:
# Initialize Snowpark session
session = Session.builder.getOrCreate()

def train_resale_model():
    try:
        # Load data
        df = session.table("GROUP4_ASG2.FINAL_DATA.PROPWISE_MASTER").to_pandas()

        target = 'RESALE_PRICE'

        numerical_features = [
            'SALE_YEAR', 'SALE_MONTH', 'MEDIAN_RESALE_PRICE',
            'YEAR_COMPLETED', 'REMAINING_LEASE_YEARS',
            'NEAREST_MRT_DISTANCE_M', 'NEAREST_MALL_M'
        ]
        
        categorical_features = ['TOWN', 'FLAT_TYPE']
        
        all_features = numerical_features + categorical_features
        
        X = df[all_features]
        y = df[target]
        
        # Remove rows with missing values
        X = X.dropna()
        y = y.loc[X.index]
        
        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
        
        # Create preprocessing pipeline
        preprocessor = ColumnTransformer(
            transformers=[
                ('num', StandardScaler(), numerical_features),
                ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
            ])
        
        # Create and train model
        model = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('regressor', GradientBoostingRegressor(
                n_estimators=50,  # Reduced for faster execution
                learning_rate=0.1,
                max_depth=3,
                random_state=42
            ))
        ])
        
        model.fit(X_train, y_train)
        
        # Make predictions
        y_pred = model.predict(X_test)
        
        # Calculate metrics
        mae = mean_absolute_error(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred)
        
        # Save model locally
        model_file = 'resale_price_model.pkl'
        joblib.dump(model, model_file)
        
        # Upload to Snowflake stage
        session.file.put(
            f'file://{model_file}',
            '@GROUP4_ASG2.FINAL_DATA.MODELS',
            overwrite=True
        )
        
        # Return metrics
        metrics = {
            'MAE': float(mae),
            'MSE': float(mse),
            'RMSE': float(rmse),
            'R2': float(r2),
            'sample_size': len(df),
            'training_size': len(X_train),
            'testing_size': len(X_test)
        }
        
        return json.dumps(metrics, indent=2)
        
    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:
# Execute the function
result = train_resale_model()
print(result)

# Output: {  
#  "MAE": 59070.25576515924,  
#  "MSE": 6675385915.321815,  
#  "RMSE": 81703.03492112037,  
#  "R2": 0.760669701641427,  
#  "sample_size": 5956379,  
#  "training_size": 3879188,  
#  "testing_size": 969797  
# }